Cover merge. Follows up on `02_cover_ablation.ipynb`, which trained and tested only on serials with a valid cover (uploaded before chapter 3, never changed) and got Recall@slots 0.289 for S+A+T, against 0.482 on all serials.

- Plan 1: is 0.289 low because of less training data, or because these serials are harder? Score the main model (trained on all serials) on the same valid-cover test serials.
- Plan 2: add covers to the model for all serials without leaking "the cover was changed later". The main model is trained on all serials; a cover adjustment is trained on valid-cover serials only.

Same protocol as `modeling/01_ml.py` and `03_compare.py`: train 2017–2021, tune on 2022, retrain on 2017–2022, score 2023 once, 2024 as a secondary check.

Inputs: `embedding/out/books.parquet` (HF dataset); valid-cover list from `data/cover/cover_status.parquet`, falling back to `cover/out/clusters/cover_clusters.parquet`; for plan 2, `data/cover/cover_emb_clip_l14.parquet` (teammate's Google Drive, `RR_project1/cover_v2/封面向量_V区块_交接/03_embeddings/`). Run from the repo root: `cd cover && uv run jupyter nbconvert --to notebook --execute --inplace 03_cover_merge.ipynb`. Numbers are also saved to `cover/out/cover_merge.json`.

In [1]:
import json, os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())   # repo root
sys.path.insert(0, str(ROOT / "modeling"))
OUT_COVER = ROOT / "cover" / "out"
import features as F

COVER_DIR = Path(os.environ.get("COVER_DIR", ROOT / "data" / "cover"))   # override with COVER_DIR if the cover files live elsewhere
STATUS_PATH = COVER_DIR / "cover_status.parquet"
CLUSTER_PATH = OUT_COVER / "clusters" / "cover_clusters.parquet"
EMB_PATH = COVER_DIR / "cover_emb_clip_l14.parquet"

CS = [1e-4, 1e-3, 1e-2, 1e-1]          # C grid for the main model, same as 01_ml.py
CS_COVER = [1e-5, 1e-4, 1e-3, 1e-2]    # C grid for the cover adjustment (cover-only models picked 1e-4, so one smaller value is added)
B = 500                                # bootstrap draws, same as 03_compare.py
SEED = 0
N_FOLDS = 5
pd.set_option("display.precision", 3)
T0 = time.time()
RES = {}                               # written to cover/out/cover_merge.json at the end

In [2]:
b, X = F.load()
sp = F.splits(b)
y, q, fid = b.top10.values.astype(int), b.quarter.values, b.fiction_id.values
trva = sp["train"] | sp["val"]

if STATUS_PATH.exists():
    st = pd.read_parquet(STATUS_PATH, columns=["fiction_id", "status"]).set_index("fiction_id").status
    status = pd.Series(fid).map(st).fillna("missing").values
    VALID_SOURCE = "data/cover/cover_status.parquet"
else:
    cl = pd.read_parquet(CLUSTER_PATH)
    status = np.where(np.isin(fid, cl.fiction_id.values), "valid", "not_valid")
    VALID_SOURCE = "cover/out/clusters/cover_clusters.parquet (clustering used valid covers only, so its list is the valid-cover list)"
valid = status == "valid"
assert valid.sum() == 19006, valid.sum()
print("Valid-cover list from:", VALID_SOURCE)

rows = []
for k in ["train", "val", "test", "recent"]:
    m = sp[k]
    rows.append({"split": k, "n": int(m.sum()), "n_valid": int((m & valid).sum()),
                 "hit_rate_all": y[m].mean(), "hit_rate_valid": y[m & valid].mean(), "hit_rate_other": y[m & ~valid].mean()})
RES["split_counts"] = rows
pd.DataFrame(rows).set_index("split")

Valid-cover list from: data/cover/cover_status.parquet


,n,n_valid,hit_rate_all,hit_rate_valid,hit_rate_other
split,,,,,
train,17100,4236,0.101,0.077,0.108
val,5700,2927,0.100,0.059,0.144
test,8075,4507,0.100,0.056,0.156
recent,12165,7335,0.100,0.063,0.157


Plan 1. Two S+A+T models with identical features; only the training serials differ: all serials (the repo's main model `lr_SAT`, 22,800 serials) or valid-cover serials only (the ablation's setup, 7,163 serials, almost all 2020–2022). Both are scored on the same test serials. A rule (popular genres first) is shown for reference. Recall@slots depends on the hit rate, so "x random" (Recall@slots divided by the hit rate) is shown too.

In [3]:
lr = lambda C: make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000))

def fit_select(Z, mask):
    # As in 01_ml.py: try each C on train∩mask, pick C by AUC on val∩mask, refit on (train+val)∩mask. Returns (C, validation metrics, scores for all serials)
    tr, va = sp["train"] & mask, sp["val"] & mask
    val = {C: F.metrics(y[va], lr(C).fit(Z[tr], y[tr]).decision_function(Z[va]), q[va]) for C in CS}
    C = max(val, key=lambda c: val[c]["auc"])
    return C, val[C], lr(C).fit(Z[tr | va], y[tr | va]).decision_function(Z)

def evaluate(score, mask):
    m = F.metrics(y[mask], score[mask], q[mask])
    rate = y[mask].mean()
    return {"n": int(mask.sum()), "hits": int(y[mask].sum()), "auc": m["auc"], "recall_at_slots": m["recall_at_slots"],
            "random": rate, "x_random": m["recall_at_slots"] / rate}

rng = np.random.default_rng(SEED)
def paired_boot(sa, sb, mask):
    # Resample serials within each quarter (keeps the slots-per-quarter structure); 95% interval of sa minus sb for AUC and Recall@slots
    idx = np.flatnonzero(mask)
    by_q = [idx[q[idx] == qq] for qq in np.unique(q[idx])]
    d = np.empty((B, 2))
    for i in range(B):
        ix = np.concatenate([rng.choice(g, len(g)) for g in by_q])
        ma, mb = F.metrics(y[ix], sa[ix], q[ix]), F.metrics(y[ix], sb[ix], q[ix])
        d[i] = ma["auc"] - mb["auc"], ma["recall_at_slots"] - mb["recall_at_slots"]
    lo, hi = np.percentile(d, [2.5, 97.5], axis=0)
    pa, pb = F.metrics(y[mask], sa[mask], q[mask]), F.metrics(y[mask], sb[mask], q[mask])
    return {"auc_diff": pa["auc"] - pb["auc"], "auc_ci95": (lo[0], hi[0]),
            "recall_diff": pa["recall_at_slots"] - pb["recall_at_slots"], "recall_ci95": (lo[1], hi[1])}

def check(name, ok, detail):
    print(("PASS  " if ok else "FAIL  ") + name + " | " + str(detail))

In [4]:
Z_SAT = F.block(X, "SAT")
t = time.time()
C_all, val_all, s_all = fit_select(Z_SAT, np.ones(len(b), bool))   # trained on all serials = repo main model
C_v, val_v, s_v = fit_select(Z_SAT, valid)                          # trained on valid-cover serials only = ablation setup
print(f"All serials: C={C_all}, validation AUC {val_all['auc']:.3f} | valid-cover only: C={C_v}, validation AUC {val_v['auc']:.3f} ({time.time() - t:.0f} s)")

# Rule baseline: popular genres first (same as 01_ml.py)
tag_cols = [i for i, c in enumerate(b.attrs["cols"]["S"]) if c.startswith("tag_")]
TG = X["S"][:, tag_cols]
tag_rate = (TG[trva] * y[trva, None]).sum(0) / np.maximum(TG[trva].sum(0), 1)
s_rule = (TG @ tag_rate) / np.maximum(TG.sum(1), 1)

te_all = F.metrics(y[sp["test"]], s_all[sp["test"]], q[sp["test"]])
te_v = F.metrics(y[sp["test"] & valid], s_v[sp["test"] & valid], q[sp["test"] & valid])
check("all serials, all test serials: reproduces the repo main model (AUC 0.887, recall 0.482)",
      abs(te_all["auc"] - 0.887) < 0.002 and abs(te_all["recall_at_slots"] - 0.482) < 0.002,
      f"AUC {te_all['auc']:.3f}, recall {te_all['recall_at_slots']:.3f}")
check("valid-cover only, valid-cover test serials: reproduces the ablation's 0.289",
      abs(te_v["recall_at_slots"] - 0.289) < 0.002, f"AUC {te_v['auc']:.3f}, recall {te_v['recall_at_slots']:.3f}")

All serials: C=0.001, validation AUC 0.877 | valid-cover only: C=0.001, validation AUC 0.862 (2 s)
PASS  all serials, all test serials: reproduces the repo main model (AUC 0.887, recall 0.482) | AUC 0.887, recall 0.482
PASS  valid-cover only, valid-cover test serials: reproduces the ablation's 0.289 | AUC 0.858, recall 0.289


In [5]:
SUBSETS = {"2023 all": sp["test"], "2023 valid cover": sp["test"] & valid, "2023 other covers": sp["test"] & ~valid,
           "2024 valid cover": sp["recent"] & valid, "2024 all": sp["recent"]}
MODELS1 = {"Rule: popular genres first": s_rule, "S+A+T trained on valid-cover serials only": s_v, "S+A+T trained on all serials (main)": s_all}
tab1 = pd.DataFrame([{"test_set": sn, "model": mn, **evaluate(s, m)} for sn, m in SUBSETS.items() for mn, s in MODELS1.items()])
RES["plan1_table"] = tab1.to_dict("records")
tab1.set_index(["test_set", "model"])

n  hits  \
test_set          model                                                    
2023 all          Rule: popular genres first                  8075   811   
                  S+A+T trained on valid-cover serials only   8075   811   
                  S+A+T trained on all serials (main)         8075   811   
2023 valid cover  Rule: popular genres first                  4507   253   
                  S+A+T trained on valid-cover serials only   4507   253   
                  S+A+T trained on all serials (main)         4507   253   
2023 other covers Rule: popular genres first                  3568   558   
                  S+A+T trained on valid-cover serials only   3568   558   
                  S+A+T trained on all serials (main)         3568   558   
2024 valid cover  Rule: popular genres first                  7335   461   
                  S+A+T trained on valid-cover serials only   7335   461   
                  S+A+T trained on all serials (main)         7335   461   
2024 all          Rule: popular genres first                 12165  1220   
                  S+A+T trained on valid-cover serials only  12165  1220   
                  S+A+T trained on all serials (main)        12165  1220   

                                                               auc  \
test_set          model                                              
2023 all          Rule: popular genres first                 0.742   
                  S+A+T trained on valid-cover serials only  0.859   
                  S+A+T trained on all serials (main)        0.887   
2023 valid cover  Rule: popular genres first                 0.747   
                  S+A+T trained on valid-cover serials only  0.858   
                  S+A+T trained on all serials (main)        0.878   
2023 other covers Rule: popular genres first                 0.727   
                  S+A+T trained on valid-cover serials only  0.857   
                  S+A+T trained on all serials (main)        0.888   
2024 valid cover  Rule: popular genres first                 0.777   
                  S+A+T trained on valid-cover serials only  0.865   
                  S+A+T trained on all serials (main)        0.876   
2024 all          Rule: popular genres first                 0.765   
                  S+A+T trained on valid-cover serials only  0.853   
                  S+A+T trained on all serials (main)        0.870   

                                                             recall_at_slots  \
test_set          model                                                        
2023 all          Rule: popular genres first                           0.261   
                  S+A+T trained on valid-cover serials only            0.408   
                  S+A+T trained on all serials (main)                  0.482   
2023 valid cover  Rule: popular genres first                           0.209   
                  S+A+T trained on valid-cover serials only            0.289   
                  S+A+T trained on all serials (main)                  0.348   
2023 other covers Rule: popular genres first                           0.330   
                  S+A+T trained on valid-cover serials only            0.514   
                  S+A+T trained on all serials (main)                  0.570   
2024 valid cover  Rule: popular genres first                           0.210   
                  S+A+T trained on valid-cover serials only            0.345   
                  S+A+T trained on all serials (main)                  0.351   
2024 all          Rule: popular genres first                           0.282   
                  S+A+T trained on valid-cover serials only            0.414   
                  S+A+T trained on all serials (main)                  0.453   

                                                             random  x_random  
test_set          model                                                        
2023 all          Rule: popular genres first                

In [6]:
# Where the 2023 hits are, by cover status (needs cover_status.parquet)
if STATUS_PATH.exists():
    te = sp["test"]
    by_st = pd.DataFrame({"status": status[te], "top10": y[te]}).groupby("status").top10.agg(n="size", hits="sum", hit_rate="mean")
    by_st["share_of_hits"] = by_st["hits"] / by_st["hits"].sum()
    RES["test_hits_by_status"] = by_st.reset_index().to_dict("records")
    display(by_st.sort_values("hits", ascending=False))
else:
    print("No cover_status.parquet; skipping the breakdown by cover status")

,n,hits,hit_rate,share_of_hits
status,,,,
after_ch3,2551,544,0.213,0.671
valid,4507,253,0.056,0.312
no_custom_cover,995,12,0.012,0.015
image_changed,20,2,0.100,0.002
no_timestamp,2,0,0.000,0.000


In [7]:
boot1 = {sn: paired_boot(s_all, s_v, SUBSETS[sn]) for sn in ["2023 valid cover", "2024 valid cover"]}
RES["plan1_boot_all_minus_validonly"] = boot1
for sn, r in boot1.items():
    print(f"{sn}: all serials minus valid-cover only | recall {r['recall_diff']:+.3f} (95% CI {r['recall_ci95'][0]:+.3f} to {r['recall_ci95'][1]:+.3f})"
          f" | AUC {r['auc_diff']:+.3f} ({r['auc_ci95'][0]:+.3f} to {r['auc_ci95'][1]:+.3f})")

2023 valid cover: all serials minus valid-cover only | recall +0.059 (95% CI +0.004 to +0.096) | AUC +0.020 (+0.008 to +0.031)
2024 valid cover: all serials minus valid-cover only | recall +0.007 (95% CI -0.028 to +0.036) | AUC +0.011 (+0.002 to +0.018)


Plan 1, what we found:

1. Part of the gap is training data. On the same 4,507 valid-cover test serials (253 hits), training on all serials lifts Recall@slots from 0.289 to 0.348 (+5.9 points, 95% CI +0.4 to +9.6) and AUC from 0.858 to 0.878.
2. The rest is the hit rate, not difficulty. Only 5.6% of valid-cover serials are hits (10% overall), so random picks score 0.056. Relative to random the main model does better here (6.2x) than on all serials (4.8x).
3. Most hits are not valid-cover serials: 558 of the 811 hits in 2023 (69%), mostly covers added or changed after chapter 3.
4. The ablation's +7.1 points were measured against 0.289. Against 0.348, its S+A+T+V (0.360) is only 1.2 points higher. Most of that gain comes from more training data; plan 2 tests what covers add on top.
5. 2024: training on all serials adds only 0.7 points of recall (CI crosses 0), but AUC is still significantly higher (+0.011).

Plan 2. Appending the cover vector and filling zeros for serials without a valid cover would let the model learn "all zeros = cover changed after chapter 3", a group with a 21% hit rate (7% for valid covers). That is leakage. Instead, using logistic regression throughout:

1. Stage 1, main model: S+A+T trained on all serials (the plan 1 main model).
2. Stage 2, cover adjustment: a logistic regression trained on valid-cover serials only. Input is the CLIP ViT-L/14 cover vector (768-d); the main score is a fixed offset (coefficient 1), so it only learns how much to add or subtract. Cover vectors are standardized and there is no intercept, so the adjustment averages exactly 0 over training serials: covers can reorder valid-cover serials but cannot lift or sink them as a group. Main scores for training serials come from 5-fold cross-validation grouped by author, because in-sample scores are too good and would leave covers too little to explain.
3. Final score: valid cover = main score + adjustment; otherwise main score. In deployment every serial has its cover at chapter 3, so stage 2 would apply to all.

The adjustment's C is picked by AUC of main + adjustment on 2022 valid-cover serials. Two controls: the ablation's setup (S+A+T+V on valid-cover serials only, reproduces 0.360) and shuffled covers (cover vectors permuted across serials when training stage 2; a real cover signal should disappear).

In [8]:
HAVE_EMB = EMB_PATH.exists()
if not HAVE_EMB:
    print("No cover vectors found; skipping plan 2. Put these two files from the teammate's Google Drive in data/cover/ and rerun:\n"
          "  RR_project1/cover_v2/封面向量_V区块_交接/03_embeddings/cover_emb_clip_l14.parquet\n"
          "  RR_project1/cover_v2/封面向量_V区块_交接/01_status/cover_status.parquet (recommended: enables the breakdown by cover status)")
else:
    t_ = pq.read_table(EMB_PATH)
    ids = t_.column("fiction_id").to_numpy()
    E = t_.column("emb").combine_chunks().flatten().to_numpy().reshape(len(ids), -1).astype(np.float64)
    pos = pd.Series(np.arange(len(ids)), index=ids)
    V = np.zeros((len(b), E.shape[1]))
    have = valid & np.isin(fid, ids)
    check("every valid-cover serial has a cover vector", have.sum() == valid.sum(), f"{have.sum()} / {valid.sum()}")
    V[valid] = E[pos.reindex(fid[valid]).values.astype(int)]
    print("Cover vectors:", V.shape, "(only valid-cover serials have values; the others are never used)")

PASS  every valid-cover serial has a cover vector | 19006 / 19006
Cover vectors: (45245, 768) (only valid-cover serials have values; the others are never used)


In [9]:
if HAVE_EMB:
    groups = b.author_id.values

    def oof_base(fit_mask, C):
        # Serials in fit_mask: N_FOLDS folds grouped by author; each fold is scored by a main model trained on the other folds
        idx = np.flatnonzero(fit_mask)
        out = np.full(len(b), np.nan)
        for k, (a, c) in enumerate(GroupKFold(N_FOLDS).split(idx, groups=groups[idx])):
            out[idx[c]] = lr(C).fit(Z_SAT[idx[a]], y[idx[a]]).decision_function(Z_SAT[idx[c]])
        return out

    t = time.time()
    # For picking C: cross-validated scores on 2017–2021; 2022 scores from a main model trained on 2017–2021
    base_sel = oof_base(sp["train"], C_all)
    base_sel[sp["val"]] = lr(C_all).fit(Z_SAT[sp["train"]], y[sp["train"]]).decision_function(Z_SAT[sp["val"]])
    # For the final model: cross-validated scores on 2017–2022; 2023 and 2024 scores are the plan 1 main model s_all
    base_fin = oof_base(trva, C_all)
    base_fin[sp["test"] | sp["recent"]] = s_all[sp["test"] | sp["recent"]]
    print(f"Cross-validated main scores done ({time.time() - t:.0f} s)")

Cross-validated main scores done (1 s)


In [10]:
if HAVE_EMB:
    def fit_adjust(fit_mask, offset, C, Vin=None):
        # On valid-cover serials: logit = offset + standardized cover vector · w, no intercept, L2 penalty ||w||² / (2C). Returns a function giving the adjustment for any serials
        Vin = V if Vin is None else Vin
        sc = StandardScaler().fit(Vin[fit_mask])
        Vs, yy, off = sc.transform(Vin[fit_mask]), y[fit_mask], offset[fit_mask]
        def f(w):
            z = off + Vs @ w
            return (np.logaddexp(0, z) - yy * z).sum() + w @ w / (2 * C), Vs.T @ (expit(z) - yy) + w / C
        w = minimize(f, np.zeros(Vs.shape[1]), jac=True, method="L-BFGS-B", options={"maxiter": 3000}).x
        return lambda mask: sc.transform(V[mask]) @ w

    fit_sel, val_v_mask = sp["train"] & valid, sp["val"] & valid
    sel_rows = [{"C": "no cover", "val_auc_2022_valid_cover": F.metrics(y[val_v_mask], base_sel[val_v_mask], q[val_v_mask])["auc"]}]
    val_auc = {}
    for C in CS_COVER:
        adj = fit_adjust(fit_sel, base_sel, C)
        val_auc[C] = F.metrics(y[val_v_mask], base_sel[val_v_mask] + adj(val_v_mask), q[val_v_mask])["auc"]
        sel_rows.append({"C": f"{C:g}", "val_auc_2022_valid_cover": val_auc[C]})
    sel = pd.DataFrame(sel_rows)
    C_cov = max(CS_COVER, key=lambda c: val_auc[c])
    RES["plan2_cover_C_selection"] = sel.to_dict("records"); RES["plan2_cover_C"] = C_cov
    print("Cover adjustment C =", C_cov)
    display(sel.set_index("C"))

Cover adjustment C = 0.001


,val_auc_2022_valid_cover
C,
no cover,0.879
1e-05,0.879
0.0001,0.882
0.001,0.883
0.01,0.853


In [11]:
if HAVE_EMB:
    fit_fin = trva & valid
    ev = (sp["test"] | sp["recent"]) & valid
    adj = fit_adjust(fit_fin, base_fin, C_cov)
    s_two = s_all.copy()
    s_two[ev] = s_all[ev] + adj(ev)                       # valid cover: main + adjustment; others unchanged
    # Per-serial two-stage scores for the demo page (read by modeling/05_demo_data.py; parquet, not in git)
    ev_all = sp["test"] | sp["recent"]
    pd.DataFrame({"fiction_id": fid[ev_all], "lr_SATC": s_two[ev_all]}).to_parquet(OUT_COVER / "preds_cover.parquet", index=False)
    for k, yr_ in [("test", 2023), ("recent", 2024)]:
        a_ = adj(sp[k] & valid)
        print(f"Cover adjustment on {yr_} valid-cover serials: mean {a_.mean():+.3f}, sd {a_.std():.3f} (exactly 0 on training serials)")
        RES[f"plan2_adjust_mean_{yr_}"] = a_.mean()

    # Diagnostic (added after seeing the results): center the adjustment within each quarter among valid-cover serials, keeping only the reordering and removing any group-wide shift
    s_two_c = s_all.copy()
    a_ev = pd.Series(adj(ev), index=np.flatnonzero(ev))
    s_two_c[ev] = s_all[ev] + (a_ev - a_ev.groupby(q[ev]).transform("mean")).values

    # Control 1: shuffled cover vectors (breaks the link between a cover and its serial)
    perm = np.arange(len(b)); vi = np.flatnonzero(valid); perm[vi] = rng.permutation(vi)
    adj_sh = fit_adjust(fit_fin, base_fin, C_cov, Vin=V[perm])
    s_shuf = s_all.copy(); s_shuf[ev] = s_all[ev] + adj_sh(ev)
    # Control 2: the ablation's setup, S+A+T+V trained on valid-cover serials only
    _, _, s_satv_v = fit_select(np.hstack([Z_SAT, V]), valid)

    MODELS2 = {"Main model S+A+T (all serials)": s_all, "Two-stage: main + cover adjustment": s_two,
               "Diagnostic: adjustment centered per quarter": s_two_c,
               "Control: shuffled covers": s_shuf, "Control: S+A+T+V on valid covers only": s_satv_v}
    tab2 = pd.DataFrame([{"test_set": sn, "model": mn, **evaluate(s, m)} for sn, m in SUBSETS.items() if sn != "2023 other covers"
                         for mn, s in MODELS2.items() if not (mn.startswith("Control: S+A+T+V") and "valid" not in sn)])
    RES["plan2_table"] = tab2.to_dict("records")
    display(tab2.set_index(["test_set", "model"]))

Cover adjustment on 2023 valid-cover serials: mean +0.076, sd 0.339 (exactly 0 on training serials)
Cover adjustment on 2024 valid-cover serials: mean +0.122, sd 0.346 (exactly 0 on training serials)


n  hits  \
test_set         model                                                      
2023 all         Main model S+A+T (all serials)                8075   811   
                 Two-stage: main + cover adjustment            8075   811   
                 Diagnostic: adjustment centered per quarter   8075   811   
                 Control: shuffled covers                      8075   811   
2023 valid cover Main model S+A+T (all serials)                4507   253   
                 Two-stage: main + cover adjustment            4507   253   
                 Diagnostic: adjustment centered per quarter   4507   253   
                 Control: shuffled covers                      4507   253   
                 Control: S+A+T+V on valid covers only         4507   253   
2024 valid cover Main model S+A+T (all serials)                7335   461   
                 Two-stage: main + cover adjustment            7335   461   
                 Diagnostic: adjustment centered per quarter   7335   461   
                 Control: shuffled covers                      7335   461   
                 Control: S+A+T+V on valid covers only         7335   461   
2024 all         Main model S+A+T (all serials)               12165  1220   
                 Two-stage: main + cover adjustment           12165  1220   
                 Diagnostic: adjustment centered per quarter  12165  1220   
                 Control: shuffled covers                     12165  1220   

                                                                auc  \
test_set         model                                                
2023 all         Main model S+A+T (all serials)               0.887   
                 Two-stage: main + cover adjustment           0.884   
                 Diagnostic: adjustment centered per quarter  0.886   
                 Control: shuffled covers                     0.883   
2023 valid cover Main model S+A+T (all serials)               0.878   
                 Two-stage: main + cover adjustment           0.881   
                 Diagnostic: adjustment centered per quarter  0.881   
                 Control: shuffled covers                     0.868   
                 Control: S+A+T+V on valid covers only        0.870   
2024 valid cover Main model S+A+T (all serials)               0.876   
                 Two-stage: main + cover adjustment           0.878   
                 Diagnostic: adjustment centered per quarter  0.878   
                 Control: shuffled covers                     0.871   
                 Control: S+A+T+V on valid covers only        0.878   
2024 all         Main model S+A+T (all serials)               0.870   
                 Two-stage: main + cover adjustment           0.865   
                 Diagnostic: adjustment centered per quarter  0.869   
                 Control: shuffled covers                     0.867   

                                                              recall_at_slots  \
test_set         model                                                          
2023 all         Main model S+A+T (all serials)                         0.482   
                 Two-stage: main + cover adjustment                     0.481   
                 Diagnostic: adjustment centered per quarter            0.487   
                 Control: shuffled covers                               0.470   
2023 valid cover Main model S+A+T (all serials)                         0.348   
                 Two-stage: main + cover adjustment                     0.387   
                 Diagnostic: adjustment centered per quarter            0.387   
                 Control: shuffled covers                               0.324   
                 Control: S+A+T+V on valid covers only                  0.360   
2024 valid cover Main model S+A+T (all serials)                         0.351   
                 Two-stage: main + cover adjustment                     0.351   
                 Diagnostic: adjustment centered pe

In [12]:
if HAVE_EMB:
    boot2 = {}
    for sn in ["2023 valid cover", "2023 all", "2024 valid cover", "2024 all"]:
        boot2[f"two-stage minus main | {sn}"] = paired_boot(s_two, s_all, SUBSETS[sn])
    for sn in ["2023 all", "2024 all"]:
        boot2[f"centered minus main | {sn}"] = paired_boot(s_two_c, s_all, SUBSETS[sn])
    boot2["shuffled covers minus main | 2023 valid cover"] = paired_boot(s_shuf, s_all, SUBSETS["2023 valid cover"])
    boot2["valid-cover-only S+A+T+V minus main | 2023 valid cover"] = paired_boot(s_satv_v, s_all, SUBSETS["2023 valid cover"])
    RES["plan2_boot"] = boot2
    for k, r in boot2.items():
        print(f"{k}: recall {r['recall_diff']:+.3f} ({r['recall_ci95'][0]:+.3f} to {r['recall_ci95'][1]:+.3f})"
              f" | AUC {r['auc_diff']:+.3f} ({r['auc_ci95'][0]:+.3f} to {r['auc_ci95'][1]:+.3f})")

two-stage minus main | 2023 valid cover: recall +0.040 (+0.012 to +0.084) | AUC +0.003 (-0.003 to +0.008)
two-stage minus main | 2023 all: recall -0.001 (-0.017 to +0.011) | AUC -0.003 (-0.005 to -0.001)
two-stage minus main | 2024 valid cover: recall +0.000 (-0.013 to +0.036) | AUC +0.002 (-0.002 to +0.006)
two-stage minus main | 2024 all: recall -0.021 (-0.029 to -0.001) | AUC -0.005 (-0.007 to -0.003)
centered minus main | 2023 all: recall +0.005 (-0.013 to +0.015) | AUC -0.001 (-0.003 to +0.001)
centered minus main | 2024 all: recall -0.011 (-0.021 to +0.006) | AUC -0.001 (-0.003 to +0.001)
shuffled covers minus main | 2023 valid cover: recall -0.024 (-0.051 to +0.024) | AUC -0.010 (-0.016 to -0.005)
valid-cover-only S+A+T+V minus main | 2023 valid cover: recall +0.012 (-0.032 to +0.073) | AUC -0.008 (-0.020 to +0.003)


Plan 2, what we found:

1. Covers help when comparing valid-cover serials with each other, but only in 2023: Recall@slots 0.348 → 0.387 (+4.0 points, 95% CI +1.2 to +8.4). Shuffled covers lose 2.4 points instead, so the gain comes from the cover content. In 2024 there is no gain (0.351 vs 0.351), and AUC does not change significantly in either year.
2. In the ranking of all serials there is no gain: 0.482 vs 0.481 in 2023 (CI crosses 0); 0.453 → 0.432 in 2024 (−2.1 points, significant).
3. The adjustment drifts upward over time: mean 0 on training serials, +0.076 in 2023, +0.122 in 2024. Newer covers look more "hit-like" to the model, which pushes all valid-cover serials up, and they have the lowest hit rate (about 6%). Centering per quarter (a diagnostic added after seeing the results) removes most of this: +0.5 points in 2023, −1.1 in 2024, both crossing 0.
4. Even without the drift there is little to gain: valid-cover serials hold 31% of 2023 hits (253/811); about 10 more hits among them is under 1 point over all serials.
5. The ablation's setup against the main model trained on all serials: +1.2 points, CI crosses 0 (−3.2 to +7.3).

Recommendation: keep the main model at S+A+T without covers, and present covers as an exploratory result: some signal (+4 points among valid-cover serials in 2023), not stable, and usable for only about 40% of serials (19,006 / 45,245). The analysis also confirms the leakage: covers added or changed after chapter 3 come with a 21% hit rate.

In [13]:
def clean(o):
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, (np.floating, float)): return round(float(o), 4)
    if isinstance(o, np.integer): return int(o)
    return o
RES.update({"valid_source": VALID_SOURCE, "have_cover_embeddings": HAVE_EMB, "main_C": C_all, "validonly_C": C_v})
(OUT_COVER / "cover_merge.json").write_text(json.dumps(clean(RES), ensure_ascii=False, indent=1))
print(f"Wrote cover/out/cover_merge.json | {(time.time() - T0) / 60:.1f} min in total")

Wrote cover/out/cover_merge.json | 0.7 min in total
